# Descargar velas de 5m de OKX (para backtest del Cerebro)
Ejecuta las celdas en orden (Entorno de ejecución > Ejecutar todo). Descarga perpetuos USDT de OKX con ccxt (API pública, **sin claves**) y guarda el avance en tu Google Drive, así que si Colab se desconecta puedes volver a ejecutar y continúa donde quedó.

Tiempo aproximado: ~3-4 min por símbolo por cada 365 días. Para probar rápido usa `DAYS = 30` y 3 símbolos.

In [ ]:
!pip -q install ccxt

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/velas_okx'
except Exception:
    OUT_DIR = '/content/velas_okx'          # sin Drive: se pierde si Colab se reinicia
os.makedirs(OUT_DIR, exist_ok=True)
print('Guardando en:', OUT_DIR)

In [ ]:
# --- Código del descargador (igual que backtesting/download_okx_5m.py) ---
import argparse
import os
import pickle
import time

import ccxt
import pandas as pd

DEFAULT_SYMBOLS = [f"{b}/USDT:USDT" for b in (
    "BTC ETH SOL BNB XRP DOGE ADA AVAX LINK LTC DOT TRX BCH NEAR ATOM UNI APT ARB OP SUI").split()]


def _fetch_with_retry(ex, symbol, tf, since, limit, tries=5):
    for k in range(tries):
        try:
            return ex.fetch_ohlcv(symbol, tf, since=since, limit=limit)
        except (ccxt.RateLimitExceeded, ccxt.NetworkError, ccxt.RequestTimeout):
            time.sleep(2 ** k)
    raise RuntimeError(f"{symbol}: demasiados fallos de red/rate limit")


def fetch_symbol(ex, symbol, days, tf="5m", limit=100, log=print):
    """Pagina hacia adelante desde now-days; descarta la vela en curso; ordena y quita duplicados."""
    tf_ms = ex.parse_timeframe(tf) * 1000
    now = ex.milliseconds()
    since = now - days * 86_400_000
    rows, last_ts, calls = [], None, 0
    while since < now:
        batch = _fetch_with_retry(ex, symbol, tf, since, limit)
        calls += 1
        if not batch:
            break
        rows += batch
        new_last = batch[-1][0]
        if last_ts is not None and new_last <= last_ts:
            break                               # sin progreso: fin de datos
        last_ts, since = new_last, new_last + tf_ms
        if calls % 100 == 0:
            log(f"  {symbol}: {len(rows)} velas, hasta {pd.to_datetime(new_last, unit='ms', utc=True):%Y-%m-%d}")
    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close", "volume"])
    df = df.drop_duplicates("ts").sort_values("ts").reset_index(drop=True)
    return df[df["ts"] + tf_ms <= now].reset_index(drop=True)   # solo velas cerradas


def quality(df, tf_ms=300_000):
    """Resumen de calidad: filas, rango y huecos (velas faltantes)."""
    if df.empty:
        return dict(rows=0, first=None, last=None, missing=0)
    expected = (df["ts"].iloc[-1] - df["ts"].iloc[0]) // tf_ms + 1
    return dict(rows=len(df), first=pd.to_datetime(df["ts"].iloc[0], unit="ms", utc=True),
                last=pd.to_datetime(df["ts"].iloc[-1], unit="ms", utc=True), missing=int(expected - len(df)))


def download_all(symbols, days, out_dir, tf="5m", ex=None, log=print):
    """Descarga todos los símbolos; guarda cada uno en out_dir/parts (si ya existe, lo reutiliza)."""
    ex = ex or ccxt.okx({"enableRateLimit": True, "options": {"defaultType": "swap"}})
    ex.load_markets()
    parts = os.path.join(out_dir, "parts")
    os.makedirs(parts, exist_ok=True)
    result = {}
    for s in symbols:
        path = os.path.join(parts, s.replace("/", "_").replace(":", "_") + f"_{days}d.pkl")
        if os.path.exists(path):
            result[s] = pickle.load(open(path, "rb"))
            log(f"{s}: reutilizado ({len(result[s])} velas)")
            continue
        if s not in ex.markets:
            log(f"{s}: no existe en OKX, se omite")
            continue
        df = fetch_symbol(ex, s, days, tf, log=log)
        for c in ("open", "high", "low", "close", "volume"):
            df[c] = df[c].astype("float32")
        pickle.dump(df, open(path, "wb"))
        result[s] = df
        q = quality(df)
        log(f"{s}: {q['rows']} velas, {q['first']:%Y-%m-%d} -> {q['last']:%Y-%m-%d}, faltan {q['missing']}")
    return result

In [ ]:
DAYS = 365                      # historia a descargar (365 o más recomendado)
SYMBOLS = DEFAULT_SYMBOLS       # o una lista propia, p.ej. ['BTC/USDT:USDT', 'ETH/USDT:USDT']
# SYMBOLS = DEFAULT_SYMBOLS[:3]; DAYS = 30   # <- descomenta para una prueba rápida

In [ ]:
data = download_all(SYMBOLS, DAYS, OUT_DIR)
print(f'\nListo: {len(data)} símbolos')

In [ ]:
import pandas as pd
resumen = pd.DataFrame({s: quality(df) for s, df in data.items()}).T
display(resumen)
malos = resumen[resumen['missing'] > 0.01 * resumen['rows']]
if len(malos): print('Atención: estos símbolos tienen más de 1% de velas faltantes:', list(malos.index))

In [ ]:
out = os.path.join(OUT_DIR, 'okx_5m.pkl')
pickle.dump(data, open(out, 'wb'))
print('Archivo:', out, f'({os.path.getsize(out)/1e6:.1f} MB)')
try:
    from google.colab import files
    files.download(out)             # también queda guardado en tu Drive
except Exception:
    pass

Después: copia `okx_5m.pkl` a tu PC y ejecuta
`python backtesting\backtest_cerebro_rl_5m.py --data okx_5m.pkl --max-symbols 20`
o sube el archivo a la sesión para entrenar el nuevo modelo.